# Bit-flip rate vs pump strength

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

Sweeps `eps_p` at fixed `alpha_sq`, rescaling `kappa_b` as
`sin(eps_p)/sin(eps_p_init) * kappa_b_init` to hold the adiabatic ratio
`kappa_b / g` constant.

**This sweep is sequential, unlike the `alpha_sq` one.** Each point is
warm-started from the previous point's Ritz vector, re-expressed in the new
eigenbasis (`transform_vectorized_state`) because the interaction-frame basis
moves with `eps_p`. Only the first point runs cold, with the larger
`m_arnoldi_first`. So the chain runs one GPU task per point, in order, rather
than fanning out.

Set `WARM_START = False` to break the chain deliberately: every point then runs
cold and independent, and they all run in parallel — more GPU-seconds, much less
wall clock.

Notebook form of `scripts/sweep_eps_p.py`; both call the same
`floquet_lindblad.pipeline.solve_point`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   — these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
— Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

In [ ]:
import time

N_A, N_B = 32, 12
ALPHA_SQ = 4
EPS_P_LIST = list(np.linspace(0.1, 1.1, 6))
KAPPA_B_INIT = 5 / 10.4  # ats.KAPPA_B
WARM_START = True

SETTINGS = dict(
    n_a=N_A,
    n_b=N_B,
    alpha_sq=ALPHA_SQ,
    cheb_degree=8,
    m_arnoldi_0=60,
    margin=1e-2,
)
M_ARNOLDI_FIRST = 120   # cold-started points
M_ARNOLDI_WARM = 60     # warm-started points


def sweep(eps_p_list, settings, kappa_b_init, m_first, m_warm, warm_start):
    """Coordinator on a worker; one GPU task per point.

    Sequential when `warm_start`: point i+1 needs point i's x_ritz *and* its V
    (the eigenbasis to rotate that vector out of). Otherwise all points are
    submitted at once.
    """
    import numpy as np
    import ray

    from anb_compute import ray as acr
    from floquet_lindblad.pipeline import solve_point_safe

    eps_p_init = eps_p_list[0]

    def kappa_b_for(e):
        return float(np.sin(e) / np.sin(eps_p_init) * kappa_b_init)

    out = []
    if not warm_start:
        refs = {
            acr.submit(solve_point_safe, eps_p=float(e), kappa_b=kappa_b_for(e),
                       m_arnoldi=m_first, want_x_ritz=False,
                       num_gpus=1, num_cpus=8, **settings): float(e)
            for e in eps_p_list
        }
        pending = list(refs)
        while pending:
            ready, pending = ray.wait(pending, num_returns=1)
            ref, e = ready[0], refs[ready[0]]
            try:
                res = ray.get(ref)
            except Exception as exc:
                out.append({"eps_p": e, "error": f"{type(exc).__name__}: {exc}"})
                continue
            out.append(res)
            print(f"eps_p = {e:5.3f}  "
                  + (res["error"] if "error" in res
                     else f"rate_bf = {res['rate_bf'].real:.6e}"))
    else:
        prev = None
        for e in eps_p_list:
            warm = prev is not None
            ref = acr.submit(
                solve_point_safe,
                eps_p=float(e), kappa_b=kappa_b_for(e),
                m_arnoldi=m_warm if warm else m_first,
                x0=prev["x_ritz"] if warm else None,
                V_prev=prev["V"] if warm else None,
                want_x_ritz=True, want_V=True,
                num_gpus=1, num_cpus=8, **settings,
            )
            try:
                res = ray.get(ref)
            except Exception as exc:
                out.append({"eps_p": float(e),
                            "error": f"{type(exc).__name__}: {exc}"})
                prev = None  # chain broken, next point restarts cold
                continue
            print(f"eps_p = {e:5.3f}  "
                  + (res["error"] if "error" in res
                     else f"rate_bf = {res['rate_bf'].real:.6e}"
                          f"  (warm={res['warm_start']})"))
            prev = res if "error" not in res else None
            # V and x_ritz only existed to feed the next point.
            out.append({k: v for k, v in res.items() if k not in ("V", "x_ritz")})

    out.sort(key=lambda d: d["eps_p"])
    return out


t0 = time.time()
results = acr.run(
    sweep, EPS_P_LIST, SETTINGS, KAPPA_B_INIT,
    M_ARNOLDI_FIRST, M_ARNOLDI_WARM, WARM_START,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
      f"for {len(results)} point(s)")

### Results

In [ ]:
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]

eps_p = np.array([r["eps_p"] for r in ok])
bf_list = np.array([r["rate_bf"] for r in ok])
kappa_b_list = np.array([r["kappa_b"] for r in ok])

print(f"{'eps_p':>7} {'kappa_b':>10} {'rate_bf':>14} {'res_rel':>11} {'warm':>5}")
for r in ok:
    print(f"{r['eps_p']:7.3f} {r['kappa_b']:10.5f} {r['rate_bf'].real:14.6e} "
          f"{r['res']['res_rel']:11.2e} {str(r['warm_start']):>5}")
for r in bad:
    print(f"{r['eps_p']:7.3f}  FAILED: {r['error']}")

### Bit-flip rate vs pump strength

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4.0), constrained_layout=True)
ax.scatter(eps_p, bf_list.real, zorder=3)
ax.set_yscale("log")
ax.set_xlabel(r"$\varepsilon_p$")
ax.set_ylabel("bit-flip rate (GHz)")
ax.grid(alpha=0.3)
plt.show()

### Adiabatic ratio

In [ ]:
# Adiabaticity check: kappa_b / g is constant by construction (kappa_b is
# rescaled as sin(eps_p), like g). g2 = J_1(eps_p) ... is not the coupling of
# this model and drifts against it. g comes back in each point's `params`, so
# the constants stay in ats.py. Both are in the model's units (no MHz
# conversion: it used to be applied to g2 only, not to kappa_b).
g = np.array([r["params"]["g"] for r in ok])
ratio = kappa_b_list / g

fig, ax = plt.subplots(figsize=(5.5, 3.4), constrained_layout=True)
ax.plot(eps_p, ratio, "o-")
ax.set_xlabel(r"$\varepsilon_p$")
ax.set_ylabel(r"$\kappa_b / g$")
ax.set_title("adiabatic ratio along the sweep")
ax.grid(alpha=0.3)
plt.show()

print("kappa_b / g :", np.round(ratio, 4))